# 04 - Embeddings Notebook (updated 08/02)

Changed this notebook so that I incorporate any pre-processing steps directly into the notebook from raw data to embeddings.  

Single-turn:  built from the WildGuardMix raw files pulled from HF used the same pre-processing steps from 06_flo_baseline.  Then created embeddings with GPT-2, Qwen3, and Nomic.
- drop missing labels
- remove deduped prompts
- SEED 1234
- 15% stratified val set

Multi-turn: built from the 

### Load Dependencies and Data

In [ ]:
# Set dependencies
import pandas as pd
import numpy as np
import torch
from pyprojroot import here
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm


# Project path anchors
REPO_ROOT = here()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"
MODEL_DIR = REPO_ROOT / "data" / "models"

SEED = 1234


### Single-turn: Build from Raw WildGuardMix

In [ ]:
# Load raw WildGuardMix (its own train/test partition; test stays locked)
df_train = pd.read_parquet(RAW_DATA_DIR / "wildguardmix" / "train" / "wildguard_train.parquet")
df_test = pd.read_parquet(RAW_DATA_DIR / "wildguardmix" / "test" / "wildguard_test.parquet")

print("raw train:", df_train.shape)
print("raw test :", df_test.shape)
df_train.head(3)


In [ ]:
# Mirrors 06_flo_baseline exactly

def build_singleturn(df):
    df = df.dropna(subset=["prompt_harm_label"]).drop_duplicates(subset="prompt").copy()
    df["harm"] = df["prompt_harm_label"] == "harmful"
    return df

train_df = build_singleturn(df_train)
test_df = build_singleturn(df_test)

train_part, val_part = train_test_split(
    train_df, test_size=0.15, random_state=SEED, stratify=train_df["harm"],
)

wgm_aligned = pd.concat([
    pd.DataFrame({"conversation": train_part["prompt"], "harm": train_part["harm"], "split": "train"}),
    pd.DataFrame({"conversation": val_part["prompt"],   "harm": val_part["harm"],   "split": "val"}),
    pd.DataFrame({"conversation": test_df["prompt"],    "harm": test_df["harm"],    "split": "test"}),
], ignore_index=True)
wgm_aligned["conversation_id"] = wgm_aligned.index.astype(str)

print(wgm_aligned["split"].value_counts())

In [ ]:
# Match 06_flo_baseline exactly: 40,674 / 7,178 / 1,699, ~51.5% harmful

counts = wgm_aligned["split"].value_counts()
assert counts["train"] == 40674, counts
assert counts["val"] == 7178, counts
assert counts["test"] == 1699, counts

print(wgm_aligned.groupby("split")["harm"].mean())

assert not wgm_aligned["conversation"].duplicated().any()
assert not wgm_aligned["conversation"].str.strip().eq("").any()

In [ ]:
wgm_aligned.to_parquet(PROCESSED_DATA_DIR / "wildguardmix_aligned.parquet", index=False)

### Single-Turn: Generate embeddings

#### GPT-2 Embeddings


In [ ]:
# GPT -2 embeddings as baseline

# check to make sure the file doesn't exist
# this setup is for a GPU, CPU's will take extra long

out_path = PROCESSED_DATA_DIR / "wildguardmix_aligned_emb_gpt2.npy"

if not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModel.from_pretrained("gpt2").to(device)
    model.eval()

    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(wgm_aligned), 16)):
            batch = wgm_aligned["conversation"].iloc[i:i + 16].tolist()
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
            outputs = model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1)
            pooled = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            all_embeddings.append(pooled.cpu().numpy())

    np.save(out_path, np.concatenate(all_embeddings, axis=0))


#### Qwen 3

In [ ]:
# Qwen 3

# check to make sure the file doesn't exist
# this setup is for a GPU, CPU's will take extra long

out_path = PROCESSED_DATA_DIR / "wildguardmix_aligned_emb_qwen3.npy"

if not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)
    embeddings = model.encode(
        wgm_aligned["conversation"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(out_path, embeddings)


#### Nomic


In [ ]:
out_path = PROCESSED_DATA_DIR / "wildguardmix_aligned_emb_nomic.npy"

if not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True, device=device)
    model.max_seq_length = 8192
    embeddings = model.encode(
        ("classification: " + wgm_aligned["conversation"]).tolist(),
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(out_path, embeddings)


### Review and check the embeddings

In [ ]:
# load embeddings
emb_gpt2 = np.load(PROCESSED_DATA_DIR / "wildguardmix_aligned_emb_gpt2.npy")
emb_qwen3 = np.load(PROCESSED_DATA_DIR / "wildguardmix_aligned_emb_qwen3.npy")
emb_nomic = np.load(PROCESSED_DATA_DIR / "wildguardmix_aligned_emb_nomic.npy")

# check
for name, emb in [("gpt2", emb_gpt2), ("qwen3", emb_qwen3), ("nomic", emb_nomic)]:
    assert emb.shape[0] == len(wgm_aligned), f"{name} not row-aligned with wgm_aligned!"
    assert np.isfinite(emb).all(), f"{name} has NaN/inf"
    print(f"{name:>5}: {emb.shape}")


### Generate Embeddings for Multi-turn

Waiting on Rachel's preprocessing

In [ ]:
# Load Data
mt = None

#### GPT-2

In [ ]:
out_path = PROCESSED_DATA_DIR / "multiturn_eval_emb_gpt2.npy"

if mt is not None and not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModel.from_pretrained("gpt2").to(device)
    model.eval()

    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(mt), 8)):
            batch = mt["conversation"].iloc[i:i + 8].tolist()
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
            outputs = model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1)
            pooled = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            all_embeddings.append(pooled.cpu().numpy())

    np.save(out_path, np.concatenate(all_embeddings, axis=0))



#### Qwen-3

In [ ]:
out_path = PROCESSED_DATA_DIR / "multiturn_eval_emb_qwen3.npy"

if mt is not None and not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)
    embeddings = model.encode(
        mt["conversation"].tolist(),
        batch_size=1,          # long conversations — avoid OOM
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(out_path, embeddings)


#### Nomic

In [ ]:
out_path = PROCESSED_DATA_DIR / "multiturn_eval_emb_nomic.npy"

if mt is not None and not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True, device=device)
    model.max_seq_length = 8192
    embeddings = model.encode(
        ("classification: " + mt["conversation"]).tolist(),
        batch_size=1,          # long conversations — avoid OOM
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(out_path, embeddings)


#### Review and Check

In [ ]:
if mt is not None:
    for name in ("gpt2", "qwen3", "nomic"):
        emb = np.load(PROCESSED_DATA_DIR / f"multiturn_eval_emb_{name}.npy")
        assert emb.shape[0] == len(mt), f"{name} not row-aligned with multiturn file!"
        assert np.isfinite(emb).all(), f"{name} has NaN/inf"
        print(f"{name:>5}: {emb.shape}")

